## INIT

In [0]:
import sys
import os
sys.path.append(os.path.abspath('../../'))
from utils.logger_utils import get_project_logger
from datetime import datetime
from utils.pipeline_monitoring import log_pipeline_run

dataset_name = "color_dim"
layer = "gold"

start_time = datetime.now()

logger = get_project_logger("Gold_Dimension_color")
logger.info("--- Starting Gold Layer: Creating dim_color ---")

## Creating a gold dimension view

In [0]:
start_time = datetime.now()

try:
    logger.info("Step 1/2: Running SQL to create dim_color view")
    
    sql_query = """
    CREATE OR REPLACE VIEW transport_lakehouse.gold.dim_color AS

    SELECT
        ROW_NUMBER() OVER (ORDER BY color_cd) AS color_key,
        color_cd,
        color_nm
    FROM (

        SELECT CAST(color_cd AS STRING) AS color_cd, color_nm
        FROM transport_lakehouse.silver.silver_vehicles_private

        UNION

        SELECT CAST(color_cd AS STRING) AS color_cd, color_nm
        FROM transport_lakehouse.silver.silver_vehicles_public

    ) t
    WHERE color_nm IS NOT NULL
     """
    
    spark.sql(sql_query)
    logger.info("Step 1/2: View created successfully.")

    logger.info("Step 2/2: Performing Quality Check (Row Count)")
    dim_count = spark.table("transport_lakehouse.gold.dim_color").count()
    logger.info(f"Quality Check Passed: dim_color contains {dim_count} unique colors.")

    end_time = datetime.now()

    log_pipeline_run(
        spark=spark,
        dataset_name=dataset_name,
        layer=layer,
        run_start_time=start_time,
        run_end_time=end_time,
        status="SUCCESS",
        rows_ingested=dim_count,
        error_message=None
    )

except Exception as e:
    end_time = datetime.now()
    logger.error(f"Failed to create Gold Dimension: {str(e)}")

    log_pipeline_run(
        spark=spark,
        dataset_name=dataset_name,
        layer=layer,
        run_start_time=start_time,
        run_end_time=end_time,
        status="FAILED",
        rows_ingested=0,
        error_message=str(e)
    )

    raise

logger.info("--- Gold dim_color Process Completed ---")